In [ ]:
!pip install transformers accelerate bitsandbytes torch requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/microsoft/PythonProgrammingPuzzles.git

Cloning into 'PythonProgrammingPuzzles'...
remote: Enumerating objects: 1152, done.
remote: Counting objects: 100% (475/475), done.
remote: Compressing objects: 100% (291/291), done.
remote: Total 1152 (delta 320), reused 306 (delta 184), pack-reused 677 (from 1)
Receiving objects: 100% (1152/1152), 248.36 MiB | 22.26 MiB/s, done.
Resolving deltas: 100% (744/744), done.


In [ ]:
import os
os.path.exists("PythonProgrammingPuzzles/puzzles/puzzles.json")  # should print True

True

In [ ]:
import json

with open("PythonProgrammingPuzzles/puzzles/puzzles.json", "r") as f:
    puzzles = json.load(f)

print(f"Total puzzles: {len(puzzles)}")
print("Keys in each puzzle:", puzzles[0].keys())
print("\n--- Example puzzle ---")
print(puzzles[0])

Total puzzles: 1715
Keys in each puzzle: dict_keys(['name', 'sat', 'ans_type', 'sol_header', 'sol_docstring', 'sol_bodies', 'module', 'notes', 'weight'])

--- Example puzzle ---
{'name': 'Study_1:0', 'sat': "def sat(s: str):\n    return s.count('o') == 1000 and s.count('oo') == 0", 'ans_type': 'str', 'sol_header': 'def sol():', 'sol_docstring': '    """Find a string with 1000 \'o\'s but no two adjacent \'o\'s."""', 'sol_bodies': ["    return ('h' + 'o') * 1000"], 'module': 'study.py', 'notes': '', 'weight': 1.0}


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Model loaded!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [ ]:
def make_variant_prompt(sat_code: str, notes: str) -> str:
    return f"""You are a programming puzzle designer working with Python Programming Puzzles (P3).

Each P3 puzzle is a Python function called sat() that takes an answer as input and returns True if the answer is correct. The solver's job is to find an input that makes sat() return True.

Here is an original puzzle:
{sat_code}

Description: {notes}

Generate exactly 3 variants of this puzzle at different difficulty levels:

EASY: Same concept, smaller inputs, fewer constraints, or a structural hint baked in. Noticeably simpler than the original.
MEDIUM: Close to the original in difficulty. Maybe slightly relaxed constraints or marginally smaller scale.
HARD: Same concept but harder — larger inputs, stricter constraints, or an additional requirement layered on.

Rules for all variants:
1. Must be a valid sat() function in the same format
2. Must test the same core algorithmic concept
3. Must be syntactically valid Python
4. Must still require actual logic to solve

Output in exactly this format, no extra commentary:
EASY:
<sat function here>

MEDIUM:
<sat function here>

HARD:
<sat function here>
"""

def generate_variants(sat_code: str, notes: str, max_new_tokens: int = 800) -> str:
    prompt = make_variant_prompt(sat_code, notes)
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
import re

def parse_variants(raw_output: str, name: str, notes: str) -> dict:
    entry = {
        "name": name,
        "notes": notes,
        "easy":   [],
        "medium": [],
        "hard":   []
    }

    # Strip markdown code fences before matching
    clean_output = re.sub(r"```python\n|```", "", raw_output)

    pattern = r"(EASY|MEDIUM|HARD):\s*(def sat[\s\S]*?)(?=(?:EASY|MEDIUM|HARD):|$)"
    matches = re.findall(pattern, clean_output, re.IGNORECASE)

    for label, code in matches:
        level = label.lower()
        entry[level].append({"prompt": code.strip()})

    return entry


def process_puzzle(puzzle: dict) -> dict:
    sat_code = puzzle["sat"]
    notes = puzzle.get("notes", "No description available")
    name = puzzle["name"]

    raw_output = generate_variants(sat_code, notes)
    return parse_variants(raw_output, name, notes)

In [ ]:
all_results = []

MAX_RETRIES = 3

for i, puzzle in enumerate(puzzles[:10]):
    print(f"Processing {i+1}/10: {puzzle['name']}")

    entry = None
    for attempt in range(MAX_RETRIES):
        entry = process_puzzle(puzzle)
        missing = [lvl for lvl in ["easy", "medium", "hard"] if not entry[lvl]]
        if not missing:
            print(f"  All 3 tiers parsed successfully")
            break
        print(f"  Attempt {attempt+1} failed: missing {missing}, retrying...")

    if missing:
        print(f"  GIVING UP on {puzzle['name']} after {MAX_RETRIES} attempts, skipping")
    else:
        all_results.append(entry)

with open("variants_ladder.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(f"\nSaved {len(all_results)} complete problems to variants_ladder.json")

Processing 1/10: Study_1:0
  All 3 tiers parsed successfully
Processing 2/10: Study_2:0
  All 3 tiers parsed successfully
Processing 3/10: Study_3:0
  All 3 tiers parsed successfully
Processing 4/10: Study_4:0
  Attempt 1 failed: missing ['easy', 'medium', 'hard'], retrying...
  All 3 tiers parsed successfully
Processing 5/10: Study_5:0
  Attempt 1 failed: missing ['easy', 'medium', 'hard'], retrying...
  All 3 tiers parsed successfully
Processing 6/10: Study_6:0
  All 3 tiers parsed successfully
Processing 7/10: Study_7:0
  All 3 tiers parsed successfully
Processing 8/10: Study_8:0
  All 3 tiers parsed successfully
Processing 9/10: Study_9:0
  All 3 tiers parsed successfully
Processing 10/10: Study_10:0
  All 3 tiers parsed successfully

Saved 10 complete problems to variants_ladder.json


In [ ]:
entry = all_results[0]
print(f"Problem: {entry['name']}")
print(f"Notes:   {entry['notes']}")
print()
for level in ["easy", "medium", "hard"]:
    print(f"{level.upper()}:")
    print(entry[level][0]["prompt"])
    print("-" * 40)

Problem: Study_1:0
Notes:   

EASY:
def sat(s: str):
    return s.count('a') == 5 and s.count('aa') == 0
----------------------------------------
MEDIUM:
def sat(s: str):
    return s.count('b') == 20 and s.count('bb') == 0
----------------------------------------
HARD:
def sat(s: str):
    return s.count('c') == 100 and s.count('cc') == 0 and len(s) == 200
----------------------------------------
